# Symmetry-aware vs symmetry-unaware CNN — architecture sweep (L=4, topological regime)

**Question.** Does the A_v-invariance trick (the Wilson 4-product) beat a plain CNN in
the topological phase — and can a symmetry-*unaware* CNN close the gap by getting bigger
or deeper?

**Design.** At 4 topological-regime points (L=4 OBC) we train a *family* of each:
- **SA** — `ToricCNN_gridinv` (Wilson sandwich), varying noninv depth/width, inv depth/width, kernel.
- **SU** — `GeoCNN` (same geometry-exact kernel, **no Wilson**), varying depth×width via `cnn_hidden`.

All runs share the identical training budget (OBC, dt 0.01→0.001 cosine, diag_shift 1e-3,
**250 iters**, 8192 samples / 1024 chains, dense QGT). The only structural difference within
a family is size/shape; the only difference *between* families is the Wilson invariance.

This notebook is **glob-based**: it loads whatever runs are present in
`results/arch_compare/L4/` at execution time, so re-running it picks up new jobs as they land.

In [ ]:
import json, os, glob, re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

DIR = os.path.join(os.getcwd(), "..", "results", "arch_compare", "L4")

# ---- ZOOM KNOBS for the energy learning-curve figure -----------------------
# Y_SPAN : energy window shown ABOVE each panel's lowest energy (auto-anchored per
#          panel). None -> full descent.  ~2.0 late-descent · ~0.5 hx=0.2 gap ·
#          ~0.1 tiny hx=0 gaps.
# X_FROM : first iteration to display (crop the transient). 0 = whole run.
Y_SPAN = 0.5
X_FROM = 0
# ---- LINE THICKNESS knobs (SA drawn slightly heavier so it reads on top of SU) ----
LW_SA = 1.9   # symmetry-aware (warm) line width
LW_SU = 1.4   # symmetry-unaware (purple) line width
# ---- EXCLUDE knob: drop specific runs (match on filename substring) ----------
# The thin/deep SU net cnn 2·2·2·2·2·2 (1119p) DIVERGED at two points; excluded here.
# Add/remove substrings to show/hide any run, then re-run.
EXCLUDE = [
    "hx0.0_hz0.125_cnn2-2-2-2-2-2",   # SU diverged: E=-147.9, Vscore 0.58
    "hx0.0_hz0.1_cnn2-2-2-2-2-2",     # SU diverged: E=-157.9, Vscore 0.11
    "hx0.2_hz0.1_cnn2-2-2-2-2-2",     
]

plt.rcParams.update({"figure.dpi": 300, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "legend.frameon": False})
CMAP = {"SA": plt.get_cmap("autumn_r"), "SU": plt.get_cmap("Purples")}  # SA warm, SU purple
# low end of each family's shade ramp (Purples fades to white below ~0.4, so start higher)
CMAP_LO = {"SA": 0.15, "SU": 0.42}
FAMNAME = {"SA": "symmetry-aware (ToricCNN_gridinv)", "SU": "symmetry-unaware (GeoCNN)"}

In [ ]:
def classify(cfg, name):
    a = cfg.get("arch", "")
    if a == "ToricCNN_gridinv" or name.startswith("gridinv_"): return "SA"
    if a == "GeoCNN" or name.startswith("bosonic_geocnn_"):     return "SU"
    return None

def arch_label(fam, cfg):
    if fam == "SA":
        inv = cfg.get("inv_hidden", []); inv = "·".join(str(x) for x in inv)
        return f"noninv {cfg.get('n_noninv')}×{cfg.get('noninv_channels')}, inv[{inv}], k{cfg.get('kernel_size')}"
    return "cnn " + "·".join(str(x) for x in cfg.get("cnn_hidden", []))

def load_all():
    runs = []
    for f in sorted(glob.glob(os.path.join(DIR, "*.json"))):
        if f.endswith(".curve.json"): continue
        if any(x in os.path.basename(f) for x in EXCLUDE): continue
        d = json.load(open(f)); cfg = d.get("config", {}); name = d.get("name", "")
        fam = classify(cfg, name)
        if fam is None: continue
        cf = f[:-5] + ".curve.json"
        c = (json.load(open(cf)).get("curve", {}) if os.path.exists(cf) else d.get("curve", {}))
        o = d.get("observables", {})
        hx, hz = str(cfg.get("hx")), str(cfg.get("hz"))
        runs.append(dict(fam=fam, hx=hx, hz=hz, pt=(hx, hz),
                         npar=d.get("n_params"), label=arch_label(fam, cfg),
                         step=np.asarray(c.get("step", []), float),
                         E=np.asarray(c.get("energy", []), float),
                         err=np.asarray(c.get("energy_err", []), float),
                         spread=np.asarray(c.get("energy_spread", []), float),
                         E0=o.get("E0"), Vscore=o.get("Vscore"), diverged=d.get("diverged")))
    return runs

RUNS = load_all()
POINTS = sorted({(r["hx"], r["hz"]) for r in RUNS}, key=lambda t: (float(t[0]), float(t[1])))
# color shade by param rank within each family (bigger = deeper shade)
def shade_map(fam):
    ps = sorted({r["npar"] for r in RUNS if r["fam"] == fam and r["npar"] is not None})
    lo = CMAP_LO[fam]
    return {p: CMAP[fam](lo + (0.95 - lo) * (i / max(1, len(ps) - 1))) for i, p in enumerate(ps)}
SHADE = {fam: shade_map(fam) for fam in ("SA", "SU")}
def color(r): return SHADE[r["fam"]].get(r["npar"], "0.5")

n_sa = len({r["label"] for r in RUNS if r["fam"]=="SA"}); n_su = len({r["label"] for r in RUNS if r["fam"]=="SU"})
print(f"loaded {len(RUNS)} runs  |  SA archs: {n_sa}  SU archs: {n_su}  |  points: {POINTS}")
if any(r["diverged"] for r in RUNS):
    print("DIVERGED:", [(r['hx'],r['hz'],r['label']) for r in RUNS if r['diverged']])

## Final-state summary (per point, SA then SU, sorted by param count)

In [ ]:
for pt in POINTS:
    print(f"\n=== hx={pt[0]}, hz={pt[1]} ===")
    print(f"  {'fam':3} {'arch':40} {'n_par':>6} {'E_final':>10} {'Vscore':>10}")
    rs = [r for r in RUNS if (r['hx'], r['hz']) == pt]
    for r in sorted(rs, key=lambda r: (r['fam'], r['npar'] or 0)):
        E = f"{r['E'][-1]:10.3f}" if len(r['E']) else f"{'—':>10}"
        vs = f"{r['Vscore']:10.2e}" if isinstance(r['Vscore'], (int, float)) else f"{'—':>10}"
        print(f"  {r['fam']:3} {r['label']:40} {str(r['npar']):>6} {E} {vs}")
    # best (lowest) SA vs best SU final energy
    def best(fam):
        v = [r['E'][-1] for r in rs if r['fam']==fam and len(r['E'])]
        return min(v) if v else None
    bsa, bsu = best("SA"), best("SU")
    if bsa is not None and bsu is not None:
        print(f"  -> best SA={bsa:.3f}  best SU={bsu:.3f}  gap(SU-SA)={bsu-bsa:+.3f}")

## Energy learning curves — SA (warm) vs SU (purple), shaded by size

One panel per point; every architecture overlaid. Warm = symmetry-aware, purple =
symmetry-unaware; darker shade = more parameters. Tune `Y_SPAN`/`X_FROM` above and
re-run to zoom the converged tail.

In [ ]:
def grid(npan):
    ncol = 3; nrow = int(np.ceil((npan + 2) / ncol))
    fig, ax = plt.subplots(nrow, ncol, figsize=(4.4*ncol, 3.3*nrow)); return fig, ax.ravel()

def plot_panels(key, ylabel, transform, logy=False, zoom=False):
    fig, axes = grid(len(POINTS))
    for ax, pt in zip(axes, POINTS):
        rs = sorted([r for r in RUNS if (r['hx'], r['hz']) == pt],
                    key=lambda r: (r['fam'], r['npar'] or 0))
        present = []
        for r in rs:
            y = transform(r)
            if y is None or not len(y): continue
            (ax.semilogy if logy else ax.plot)(r['step'], y, color=color(r),
                lw=(LW_SA if r['fam']=="SA" else LW_SU), alpha=0.95); present.append(r)
        if X_FROM: ax.set_xlim(left=X_FROM)
        if zoom and Y_SPAN is not None and present:
            gmin = min(float(np.nanmin(r['E'])) for r in present if len(r['E']))
            ax.set_ylim(gmin - 0.08*Y_SPAN, gmin + Y_SPAN)
        ax.set_title(f"hx={pt[0]}, hz={pt[1]}"); ax.set_xlabel("iteration"); ax.set_ylabel(ylabel)
    # two legend panels: SA and SU, entries sorted by param count
    for j, fam in enumerate(("SA", "SU")):
        axl = axes[len(POINTS) + j]; axl.axis("off")
        seen = {}
        for r in sorted([r for r in RUNS if r['fam']==fam], key=lambda r: r['npar'] or 0):
            seen.setdefault(r['label'], (color(r), r['npar']))
        h = [plt.Line2D([], [], color=c, lw=(LW_SA if fam=="SA" else LW_SU),
                        label=f"{lbl}  ({np:,}p)") for lbl,(c,np) in seen.items()]
        if h: axl.legend(handles=h, loc="center", fontsize=7.5, title=FAMNAME[fam])
    for k in range(len(POINTS)+2, len(axes)): axes[k].axis("off")
    return fig

fig = plot_panels("E", "E", lambda r: r['E'], logy=False, zoom=True)
fig.suptitle(f"Energy learning curves  (y-zoom: min+{Y_SPAN})" if Y_SPAN is not None
             else "Energy learning curves (full descent)", y=1.005, fontsize=12)
fig.tight_layout(); plt.savefig("_lc_energy.png", bbox_inches="tight", dpi=300); plt.show()

## Convergence quality — energy spread (√Var H) vs iteration, log scale

Spread → 0 for an exact eigenstate. The sharpest SA-vs-SU discriminator once the
energies nearly tie.

In [ ]:
fig = plot_panels("spread", "energy spread", lambda r: np.abs(r['spread'])+1e-9, logy=True, zoom=False)
fig.suptitle("Convergence quality — energy spread vs iteration", y=1.005, fontsize=12)
fig.tight_layout(); plt.savefig("_lc_spread.png", bbox_inches="tight", dpi=300); plt.show()

## Focused comparison — both fields on (hx=0.2, hz=0.1): energy curve + variance side by side

The single point where both fields are on and the SA-vs-SU gap is largest. **Left:** energy
learning curve, styled like the 2D-paper figure with a **shaded band = per-step energy
uncertainty**. **Right:** the energy variance Var(H)=spread² vs iteration (log scale) for the
same selected runs.

Knobs (edit, then re-run this cell):
- `SELECT_SA` / `SELECT_SU` — lists of **indices** (printed below) to show only chosen archs;
  `[]` = show all. e.g. `SELECT_SU = [0, 4]`.
- `BAND` — `"err"` (statistical error of the mean, as in the paper), `"spread"` (√Var H, wider),
  or `None`. `BAND_ALPHA` sets its opacity.
- Reuses `Y_SPAN` / `X_FROM` / `LW_SA` / `LW_SU` from the config cell.

In [ ]:
FOCUS_PT   = ("0.2", "0.1")   # both fields on
SELECT_SA  = [0, 1, 2]               # indices into the SA list printed below; [] = all
SELECT_SU  = []               # indices into the SU list printed below; [] = all
BAND       = "err"            # "err"=energy_err (std err of mean) · "spread"=sqrt(VarH) · None
BAND_ALPHA = 0.25

sa_f = sorted([r for r in RUNS if r['fam']=="SA" and r['pt']==FOCUS_PT], key=lambda r: r['npar'] or 0)
su_f = sorted([r for r in RUNS if r['fam']=="SU" and r['pt']==FOCUS_PT], key=lambda r: r['npar'] or 0)
print(f"hx={FOCUS_PT[0]}, hz={FOCUS_PT[1]} — indices for SELECT_SA / SELECT_SU ([] = all):")
for i, r in enumerate(sa_f): print(f"   SA[{i}]  {r['label']:40} ({r['npar']:,}p)")
for i, r in enumerate(su_f): print(f"   SU[{i}]  {r['label']:40} ({r['npar']:,}p)")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.3))   # left: energy | right: variance
def _draw(rs, sel, fam):
    idxs = [i for i in (sel if sel else range(len(rs))) if 0 <= i < len(rs)]
    shown = []
    for i in idxs:
        r = rs[i]; y = r['E']; lw = LW_SA if fam == "SA" else LW_SU; z = 3 if fam == "SA" else 2
        # left: energy learning curve (+ uncertainty band)
        axL.plot(r['step'], y, color=color(r), lw=lw, zorder=z,
                 label=f"{fam}[{i}] {r['label']} ({r['npar']:,}p)")
        if BAND:
            b = r['err'] if BAND == "err" else np.abs(r['spread'])
            if len(b) == len(y):
                axL.fill_between(r['step'], y - b, y + b, color=color(r), alpha=BAND_ALPHA, lw=0)
        # right: energy variance  Var(H) = spread^2  (log scale)
        v = np.abs(r['spread'])**2
        if len(v): axR.semilogy(r['step'], v + 1e-12, color=color(r), lw=lw, zorder=z)
        shown.append(r)
    return shown
shown = _draw(sa_f, SELECT_SA, "SA") + _draw(su_f, SELECT_SU, "SU")
for ax in (axL, axR):
    if X_FROM: ax.set_xlim(left=X_FROM)
    ax.set_xlabel("iteration")
if Y_SPAN is not None and shown:
    gmin = min(float(np.nanmin(r['E'])) for r in shown); axL.set_ylim(gmin - 0.08*Y_SPAN, gmin + Y_SPAN)
axL.set_ylabel("E"); axL.set_title("Learning Curves" + (f"  (band = energy_{BAND})" if BAND else ""))
axR.set_ylabel("energy variance  Var(H)"); axR.set_title("energy variance (log)")
h, l = axL.get_legend_handles_labels()
fig.legend(h, l, fontsize=7, loc="center left", bbox_to_anchor=(0.91, 0.5))   # shared, outside right
fig.suptitle(f"hx={FOCUS_PT[0]}, hz={FOCUS_PT[1]} — SA (warm) vs SU (purple)", y=1.02)
fig.tight_layout(rect=(0, 0, 0.9, 1)); plt.savefig("_lc_focus.png", bbox_inches="tight", dpi=300); plt.show()

## Notes

- **Glob-based & incremental:** re-running the notebook reloads `results/arch_compare/L4/`,
  so panels fill in as the 60 jobs (32 SU + 28 SA) complete.
- Warm = SA (Wilson), purple = SU (no Wilson); darker = more parameters. If bigger/deeper
  SU curves do **not** reach the SA band, size is not a substitute for the invariance.
- Data provenance: `results/arch_compare/L4/`. Figures: `_lc_energy.png`, `_lc_spread.png`.